In [1]:
from eki_dev.aws_service import AwsService
from aws_cluster import pest_cluster
from aws_cluster.cluster_utils import (check_resource_creation_status,
check_stack_creation_status)



In [2]:
path_yaml_config = 's3://scratch-marco/parameters.yaml'

In [2]:


pest_cluster.create_pest_cluster_stack(path_yaml_config)

{'StackId': 'arn:aws:cloudformation:us-west-1:054507568115:stack/PestClusterInfrastructure/dc190630-89b8-11ef-b87f-020f1dd2931f',
 'ResponseMetadata': {'RequestId': 'bba8598e-9d24-4f3e-bfac-f453b2f83eb4',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'bba8598e-9d24-4f3e-bfac-f453b2f83eb4',
   'date': 'Sun, 13 Oct 2024 23:14:13 GMT',
   'content-type': 'text/xml',
   'content-length': '395',
   'connection': 'keep-alive'},
  'RetryAttempts': 0}}

In [2]:
check_resource_creation_status("PestClusterInfrastructure")

Stack PestClusterInfrastructure does not exist


In [3]:
check_stack_creation_status("PestClusterInfrastructure")

Stack PestClusterInfrastructure does not exist


In [ ]:
# ecs.client.update_service(
#     cluster='PestCluster',
#     service='agent_private_net',
#     desiredCount=0)

In [3]:
pest_cluster.create_main_task(path_yaml_config)

{'tasks': [{'attachments': [],
   'capacityProviderName': 'Infra-ECS-Cluster-PestCluster-a8643df8-EC2CapacityProvider-HxlrJRzZVZ1l',
   'clusterArn': 'arn:aws:ecs:us-west-1:054507568115:cluster/PestCluster',
   'containers': [],
   'cpu': '1024',
   'createdAt': datetime.datetime(2024, 10, 13, 18, 7, 59, 236000, tzinfo=tzlocal()),
   'desiredStatus': 'RUNNING',
   'enableExecuteCommand': False,
   'group': 'family:pest_host',
   'lastStatus': 'PROVISIONING',
   'launchType': 'EC2',
   'memory': '3072',
   'overrides': {'containerOverrides': [{'name': 'model',
      'command': ['pest_hp', 'wwgfm_hist_2024', '/h', ':4004'],
      'memory': 3072}],
    'inferenceAcceleratorOverrides': []},
   'tags': [{'key': 'project_tag', 'value': 'ww_2024'},
    {'key': 'user', 'value': 'marco.maneta'}],
   'taskArn': 'arn:aws:ecs:us-west-1:054507568115:task/PestCluster/f65a622afc4e434dbfdaef14d7a550bb',
   'taskDefinitionArn': 'arn:aws:ecs:us-west-1:054507568115:task-definition/pest_host:5',
   'versi

In [4]:
cf = AwsService.from_service('cloudformation')
stack_resources = cf.client.describe_stack_resources(StackName="PestClusterInfrastructure")
for resource in stack_resources['StackResources']:
    if resource['ResourceType'] == "AWS::ElasticLoadBalancingV2::TargetGroup":
        target_group_arn = resource['PhysicalResourceId']
        print(target_group_arn)
        

ClientError: An error occurred (ValidationError) when calling the DescribeStackResources operation: Stack with id PestClusterInfrastructure does not exist

In [12]:
main_task = cf.client.describe_tasks(cluster="PestCluster", tasks=['arn:aws:ecs:us-west-1:054507568115:task/PestCluster/f65a622afc4e434dbfdaef14d7a550bb'])


In [18]:
main_task

{'tasks': [{'attachments': [],
   'attributes': [{'name': 'ecs.cpu-architecture', 'value': 'x86_64'}],
   'availabilityZone': 'us-west-1c',
   'capacityProviderName': 'Infra-ECS-Cluster-PestCluster-a8643df8-EC2CapacityProvider-HxlrJRzZVZ1l',
   'clusterArn': 'arn:aws:ecs:us-west-1:054507568115:cluster/PestCluster',
   'connectivity': 'CONNECTED',
   'connectivityAt': datetime.datetime(2024, 10, 13, 18, 10, 16, 300000, tzinfo=tzlocal()),
   'containerInstanceArn': 'arn:aws:ecs:us-west-1:054507568115:container-instance/PestCluster/45479f3eeec740959e36a01f125834d1',
   'containers': [{'containerArn': 'arn:aws:ecs:us-west-1:054507568115:container/PestCluster/f65a622afc4e434dbfdaef14d7a550bb/13ec9e27-8ac3-4449-baf4-428dd4a682e5',
     'taskArn': 'arn:aws:ecs:us-west-1:054507568115:task/PestCluster/f65a622afc4e434dbfdaef14d7a550bb',
     'name': 'model',
     'image': '054507568115.dkr.ecr.us-west-1.amazonaws.com/ww_2024:dev',
     'imageDigest': 'sha256:0198069cc2cb5d2b7eb95c461ffcdbbc81213

In [16]:
main_task['tasks'][0]['containerInstanceArn']

'arn:aws:ecs:us-west-1:054507568115:container-instance/PestCluster/45479f3eeec740959e36a01f125834d1'

In [20]:
main_container = cf.client.describe_container_instances(cluster="PestCluster", 
                                       containerInstances=[main_task['tasks'][0]['containerInstanceArn']])

In [23]:
main_instance = main_container['containerInstances'][0]['ec2InstanceId']

In [22]:
cf = AwsService.from_service('ec2')

In [26]:
main_instance_ip = cf.client.describe_instances(InstanceIds = [main_instance])

In [32]:
main_instance_ip = main_instance_ip['Reservations'][0]['Instances'][0]['PrivateIpAddress']

In [36]:
cf = AwsService.from_service('ecs')

In [45]:
target_group_arn

'arn:aws:elasticloadbalancing:us-west-1:054507568115:targetgroup/PestMain/989eb9bf9ee12230'

In [46]:
cf = AwsService.from_service('elbv2')
cf.client.register_targets(TargetGroupArn=target_group_arn,
                           Targets=[{
                               'Id': main_instance_ip,
                               'Port': 4004,
                           }])

{'ResponseMetadata': {'RequestId': '8633782f-f43d-4851-b8cd-832c476333e3',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '8633782f-f43d-4851-b8cd-832c476333e3',
   'content-type': 'text/xml',
   'content-length': '253',
   'date': 'Mon, 14 Oct 2024 02:19:12 GMT'},
  'RetryAttempts': 0}}

<bound method ClientCreator._create_api_method.<locals>._api_call of <botocore.client.ElasticLoadBalancingv2 object at 0x1206e7460>>

In [3]:
pest_cluster.terminate_cluster()

{'ResponseMetadata': {'RequestId': '78dd4380-f6f3-467b-b5fe-a61bdbc4789c',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '78dd4380-f6f3-467b-b5fe-a61bdbc4789c',
   'date': 'Mon, 14 Oct 2024 02:49:45 GMT',
   'content-type': 'text/xml',
   'content-length': '212',
   'connection': 'keep-alive'},
  'RetryAttempts': 0}}